# Tutorial 4: Molecule (Drug) Embeddings

This notebook covers generating embeddings for **small molecules / drugs**
with `embpy`. Molecules are represented as **SMILES** strings.

## Available molecule embedding models

### Transformer-based models

| Model key | Architecture | Notes |
|---|---|---|
| `chemberta2MTR` | ChemBERTa-2 (MTR) | Transformer, multi-task regression head |
| `chemberta2MLM` | ChemBERTa-2 (MLM) | Transformer, masked-language-model head |
| `molformer_base` | MolFormer | Large-scale molecular transformer |

### GNN-based models

| Model key | Architecture | Notes |
|---|---|---|
| `minimol` | MiniMol (GINE) | 10M-param message-passing GNN, 512-dim fingerprints. `pip install minimol` |
| `mhg_gnn` | MHG-GNN (GIN + MHG decoder) | Hypergraph autoencoder, can **decode** embeddings back to SMILES. Install from HuggingFace. |
| `mole` | MolE (DeBERTa-based Graph Transformer) | Pre-trained on 842M molecules. **Weights not yet publicly released.** |

### RDKit fingerprints (non-learned baselines)

| Model key | Type | Notes |
|---|---|---|
| `rdkit_fp` | RDKit topological | Binary (0/1), 2048-dim |
| `morgan_fp` / `morgan_count_fp` | Morgan / ECFP | Binary or count-based |
| `maccs_fp` | MACCS keys | Binary, 167-dim |
| `atom_pair_fp` / `atom_pair_count_fp` | Atom pair | Binary or count-based |
| `torsion_fp` / `torsion_count_fp` | Topological torsion | Binary or count-based |

If you have a drug *name* instead of a SMILES, use the `DrugResolver`
(see [Tutorial 1](01_identifiers_and_preprocessing.ipynb)) to convert it first,
or let `embed_molecule` handle it automatically.

In [1]:
import logging
logging.basicConfig(level=logging.WARNING)

In [2]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

Device: cuda


## 1. Resolve drug names to SMILES

Most embedding models expect canonical SMILES. The `DrugResolver` maps
common drug names to SMILES via PubChem and automatically **standardises**
the returned SMILES (removes isotope labels, strips salts/solvents,
neutralises charges) using RDKit.

It also provides utilities for **cleaning** and **classifying** drug-column
entries before resolution:

- `DrugResolver.clean_name()` — strips salt/formulation suffixes
  (e.g. `"Almonertinib (hydrochloride)"` → `"Almonertinib"`).
- `DrugResolver.classify_name()` — categorises entries as `"drug_name"`,
  `"control"`, `"target_description"` (e.g. `"ACLY.inhibitor"`), or
  `"ambiguous"` (very short strings like `"AZ"`).

`name_to_smiles()` uses both internally: it skips controls and target
descriptions without making API calls, and falls back to the cleaned
name when the original fails.

In [3]:
from embpy.resources.drug_resolver import DrugResolver

drug_resolver = DrugResolver()

drugs = {
    "aspirin":    drug_resolver.name_to_smiles("aspirin"),
    "ibuprofen":  drug_resolver.name_to_smiles("ibuprofen"),
    "caffeine":   drug_resolver.name_to_smiles("caffeine"),
    "paclitaxel": drug_resolver.name_to_smiles("paclitaxel"),
}

for name, smi in drugs.items():
    print(f"  {name:12s} → {smi}")

[16:29:24] Initializing MetalDisconnector
[16:29:24] Running MetalDisconnector
[16:29:24] Initializing Normalizer
[16:29:24] Running Normalizer
[16:29:24] Initializing MetalDisconnector
[16:29:24] Running MetalDisconnector
[16:29:24] Initializing Normalizer
[16:29:24] Running Normalizer
[16:29:24] Running FragmentRemover
[16:29:24] Running LargestFragmentChooser
[16:29:24] Running Uncharger
[16:29:24] Initializing MetalDisconnector
[16:29:24] Running MetalDisconnector
[16:29:24] Initializing Normalizer
[16:29:24] Running Normalizer
[16:29:24] Initializing MetalDisconnector
[16:29:24] Running MetalDisconnector
[16:29:24] Initializing Normalizer
[16:29:24] Running Normalizer
[16:29:24] Running FragmentRemover
[16:29:24] Running LargestFragmentChooser
[16:29:24] Running Uncharger
[16:29:25] Initializing MetalDisconnector
[16:29:25] Running MetalDisconnector
[16:29:25] Initializing Normalizer
[16:29:25] Running Normalizer
[16:29:25] Initializing MetalDisconnector
[16:29:25] Running MetalDi

  aspirin      → CC(=O)Oc1ccccc1C(=O)O
  ibuprofen    → CC(C)Cc1ccc(C(C)C(=O)O)cc1
  caffeine     → Cn1c(=O)c2c(ncn2C)n(C)c1=O
  paclitaxel   → CC(=O)OC1C(=O)C2(C)C(O)CC3OCC3(OC(C)=O)C2C(OC(=O)c2ccccc2)C2(O)CC(OC(=O)C(O)C(NC(=O)c3ccccc3)c3ccccc3)C(C)=C1C2(C)C


[16:29:25] Initializing MetalDisconnector
[16:29:25] Running MetalDisconnector
[16:29:25] Initializing Normalizer
[16:29:25] Running Normalizer
[16:29:25] Initializing MetalDisconnector
[16:29:25] Running MetalDisconnector
[16:29:25] Initializing Normalizer
[16:29:25] Running Normalizer
[16:29:25] Running FragmentRemover
[16:29:25] Running LargestFragmentChooser
[16:29:25] Running Uncharger


In [4]:
import pandas as pd

perturbations = pd.read_csv("/lustre/groups/ml01/projects/big_perturbation/dataset_w_embeddings/perturbations.csv")
print(f"Shape: {perturbations.shape}")
perturbations.head()

Shape: (269927, 3)


,drug,gene,dataset
0,Cediranib_PCI-34051,NaN,combosciplex_with_embeddings
1,Dacinostat_Danusertib,NaN,combosciplex_with_embeddings
2,Dacinostat_Dasatinib,NaN,combosciplex_with_embeddings
3,Dacinostat_PCI-34051,NaN,combosciplex_with_embeddings
4,SRT2104_alvespimycin,NaN,combosciplex_with_embeddings


In [5]:
import re
from embpy.resources.drug_resolver import DrugResolver

resolver = DrugResolver(use_rdkit=True, sleep_sec=0.5)


def split_entry(entry):
    """Split a drug column value into individual names."""
    if pd.isna(entry):
        return []
    s = DrugResolver.clean_name(str(entry).strip())
    parts = re.split(r"[_|/]", s)
    return [p.strip() for p in parts if p.strip()]


# 1 — Collect unique individual drug names
unique_names = set()
for entry in perturbations["drug"].dropna().unique():
    unique_names.update(split_entry(entry))

# 2 — Classify each name using the resolver
classifications = {name: DrugResolver.classify_name(name) for name in unique_names}
drug_names = sorted(n for n, c in classifications.items() if c == "drug_name")
skipped = {c: sorted(n for n, cl in classifications.items() if cl == c)
           for c in ("control", "target_description", "ambiguous")}

print(f"Unique entries:         {len(unique_names)}")
print(f"  Drug names to resolve: {len(drug_names)}")
for cat, names in skipped.items():
    if names:
        print(f"  Skipped ({cat}): {names}")

# 3 — Resolve each unique drug name once
# name_to_smiles now automatically:
#  - skips controls and target descriptions
#  - strips salt suffixes as a fallback
#  - rate-limits between each API call (sleep_sec=0.3)
#  - standardises SMILES (isotopes, salts, charges)
cache = {}
for i, name in enumerate(drug_names, 1):
    smi = resolver.name_to_smiles(name)
    cache[name] = smi
    if i % 20 == 0:
        print(f"  {i}/{len(drug_names)} done...")

n_ok = sum(v is not None for v in cache.values())
print(f"\nResolved {n_ok}/{len(drug_names)} unique drugs ({100 * n_ok / len(drug_names):.1f}%)")

failed = sorted(k for k, v in cache.items() if v is None)
if failed:
    print(f"Failed ({len(failed)}): {failed}")

# 4 — Map back to dataframe using the cache
def lookup(entry):
    parts = split_entry(entry)
    s1 = cache.get(parts[0]) if len(parts) > 0 else None
    s2 = cache.get(parts[1]) if len(parts) > 1 else None
    return s1, s2

perturbations[["canonical_smiles_1", "canonical_smiles_2"]] = perturbations["drug"].apply(
    lambda x: pd.Series(lookup(x))
)

perturbations[["drug", "canonical_smiles_1", "canonical_smiles_2"]]

Unique entries:         573
  Drug names to resolve: 564
  Skipped (control): ['DMSO', 'control', 'vehicle']
  Skipped (target_description): ['ACLY.inhibitor', 'ACSS2.inhibitor', 'PDH.inhibitor']
  Skipped (ambiguous): ['AZ', 'TF', 'ZM']


[16:29:26] Initializing MetalDisconnector
[16:29:26] Running MetalDisconnector
[16:29:26] Initializing Normalizer
[16:29:26] Running Normalizer
[16:29:26] Initializing MetalDisconnector
[16:29:26] Running MetalDisconnector
[16:29:26] Initializing Normalizer
[16:29:26] Running Normalizer
[16:29:26] Running FragmentRemover
[16:29:26] Running LargestFragmentChooser
[16:29:26] Running Uncharger
[16:29:26] Initializing MetalDisconnector
[16:29:26] Running MetalDisconnector
[16:29:26] Initializing Normalizer
[16:29:26] Running Normalizer
[16:29:26] Initializing MetalDisconnector
[16:29:26] Running MetalDisconnector
[16:29:26] Initializing Normalizer
[16:29:26] Running Normalizer
[16:29:26] Running FragmentRemover
[16:29:26] Running LargestFragmentChooser
[16:29:26] Running Uncharger
[16:29:27] Initializing MetalDisconnector
[16:29:27] Running MetalDisconnector
[16:29:27] Initializing Normalizer
[16:29:27] Running Normalizer
[16:29:27] Initializing MetalDisconnector
[16:29:27] Running MetalDi

  20/564 done...


[16:31:15] Initializing MetalDisconnector
[16:31:15] Running MetalDisconnector
[16:31:15] Initializing Normalizer
[16:31:15] Running Normalizer
[16:31:15] Initializing MetalDisconnector
[16:31:15] Running MetalDisconnector
[16:31:15] Initializing Normalizer
[16:31:15] Running Normalizer
[16:31:15] Running FragmentRemover
[16:31:15] Running LargestFragmentChooser
[16:31:15] Running Uncharger
[16:31:15] Initializing MetalDisconnector
[16:31:15] Running MetalDisconnector
[16:31:15] Initializing Normalizer
[16:31:15] Running Normalizer
[16:31:15] Initializing MetalDisconnector
[16:31:15] Running MetalDisconnector
[16:31:15] Initializing Normalizer
[16:31:15] Running Normalizer
[16:31:15] Running FragmentRemover
[16:31:15] Running LargestFragmentChooser
[16:31:15] Running Uncharger
[16:31:15] Initializing MetalDisconnector
[16:31:15] Running MetalDisconnector
[16:31:15] Initializing Normalizer
[16:31:15] Running Normalizer
[16:31:15] Initializing MetalDisconnector
[16:31:15] Running MetalDi

  40/564 done...


...

[16:46:49] Initializing MetalDisconnector
[16:46:49] Running MetalDisconnector
[16:46:49] Initializing Normalizer
[16:46:49] Running Normalizer
[16:46:49] Initializing MetalDisconnector
[16:46:49] Running MetalDisconnector
[16:46:49] Initializing Normalizer
[16:46:49] Running Normalizer
[16:46:49] Running FragmentRemover
[16:46:49] Running LargestFragmentChooser
[16:46:49] Running Uncharger
[16:46:49] Initializing MetalDisconnector
[16:46:49] Running MetalDisconnector
[16:46:49] Initializing Normalizer
[16:46:49] Running Normalizer
[16:46:49] Initializing MetalDisconnector
[16:46:49] Running MetalDisconnector
[16:46:49] Initializing Normalizer
[16:46:49] Running Normalizer
[16:46:49] Running FragmentRemover
[16:46:49] Running LargestFragmentChooser
[16:46:49] Running Uncharger
[16:46:50] Initializing MetalDisconnector
[16:46:50] Running MetalDisconnector
[16:46:50] Initializing Normalizer
[16:46:50] Running Normalizer
[16:46:50] Initializing MetalDisconnector
[16:46:50] Running MetalDi

  560/564 done...


[16:46:56] Initializing MetalDisconnector
[16:46:56] Running MetalDisconnector
[16:46:56] Initializing Normalizer
[16:46:56] Running Normalizer
[16:46:56] Initializing MetalDisconnector
[16:46:56] Running MetalDisconnector
[16:46:56] Initializing Normalizer
[16:46:56] Running Normalizer
[16:46:56] Running FragmentRemover
[16:46:56] Running LargestFragmentChooser
[16:46:56] Running Uncharger
[16:46:57] Initializing MetalDisconnector
[16:46:57] Running MetalDisconnector
[16:46:57] Initializing Normalizer
[16:46:57] Running Normalizer
[16:46:57] Initializing MetalDisconnector
[16:46:57] Running MetalDisconnector
[16:46:57] Initializing Normalizer
[16:46:57] Running Normalizer
[16:46:57] Running FragmentRemover
[16:46:57] Running LargestFragmentChooser
[16:46:57] Running Uncharger
[16:46:57] Initializing MetalDisconnector
[16:46:57] Running MetalDisconnector
[16:46:57] Initializing Normalizer
[16:46:57] Running Normalizer
[16:46:57] Initializing MetalDisconnector
[16:46:57] Running MetalDi


Resolved 556/564 unique drugs (98.6%)
Failed (8): ['AZD5591', 'Anacardic', 'BRD3379', 'GNE-069', 'GNE-104', 'GSK', 'Tie2', 'Tubastatin']


,drug,canonical_smiles_1,canonical_smiles_2
0,Cediranib_PCI-34051,COc1cc2c(Oc3ccc4[nH]c(C)cc4c3F)ncnc2cc1OCCCN1C...,COc1ccc(Cn2ccc3ccc(C(=O)NO)cc32)cc1
1,Dacinostat_Danusertib,O=C(C=Cc1ccc(CN(CCO)CCc2c[nH]c3ccccc23)cc1)NO,COC(C(=O)N1Cc2[nH]nc(NC(=O)c3ccc(N4CCN(C)CC4)c...
2,Dacinostat_Dasatinib,O=C(C=Cc1ccc(CN(CCO)CCc2c[nH]c3ccccc23)cc1)NO,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...
3,Dacinostat_PCI-34051,O=C(C=Cc1ccc(CN(CCO)CCc2c[nH]c3ccccc23)cc1)NO,COc1ccc(Cn2ccc3ccc(C(=O)NO)cc32)cc1
4,SRT2104_alvespimycin,Cc1nc(-c2cccnc2)sc1C(=O)Nc1ccccc1-c1cn2c(CN3CC...,COC1C=CC=C(C)C(=O)NC2=CC(=O)C(NCCN(C)C)=C(CC(C...
...,...,...,...
269922,olaparib,O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1CCN(C(...,None
269923,palbociclib,CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n(C2C...,None
269924,venetoclax,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,None
269925,vincristine,CCC1(O)CC2CN(CCc3c([nH]c4ccccc34)C(C(=O)OC)(c3...,None


In [6]:
perturbations["_entry_class"] = perturbations["drug"].apply(
    lambda x: "nan" if pd.isna(x) else DrugResolver.classify_name(str(x).split("|")[0].split("_")[0].strip())
)

drug_rows = perturbations.loc[perturbations["_entry_class"] == "drug_name"].copy()

n_total = len(drug_rows)
n_missing_1 = int(drug_rows["canonical_smiles_1"].isna().sum())

is_combo = drug_rows["drug"].str.contains("[_|]", na=False, regex=True)
combo_rows = drug_rows.loc[is_combo]
n_combos = len(combo_rows)
n_missing_2 = int(combo_rows["canonical_smiles_2"].isna().sum())

print(f"=== Entry classification ===")
for cat in ("drug_name", "control", "target_description", "ambiguous", "nan"):
    n = int((perturbations["_entry_class"] == cat).sum())
    print(f"  {cat:>20s}: {n:,}")

print(f"\n=== First drug (drug_name rows only) ===")
print(f"Rows: {n_total:,}")
print(f"Resolved: {n_total - n_missing_1:,}/{n_total:,} ({100 * (n_total - n_missing_1) / n_total:.1f}%)")
print(f"Missing:  {n_missing_1:,}/{n_total:,} ({100 * n_missing_1 / n_total:.1f}%)")

print(f"\n=== Second drug (combos only) ===")
print(f"Combo treatments: {n_combos:,}")
if n_combos > 0:
    print(f"Resolved: {n_combos - n_missing_2:,}/{n_combos:,} ({100 * (n_combos - n_missing_2) / n_combos:.1f}%)")
    print(f"Missing:  {n_missing_2:,}/{n_combos:,} ({100 * n_missing_2 / n_combos:.1f}%)")

unresolved = drug_rows.loc[drug_rows["canonical_smiles_1"].isna(), "drug"].unique().tolist()
if unresolved:
    print(f"\nUnresolved drug names ({len(unresolved)} unique):")
    display(unresolved)

perturbations.drop(columns=["_entry_class"], inplace=True)

=== Entry classification ===
             drug_name: 625
               control: 17
    target_description: 6
             ambiguous: 2
                   nan: 269,277

=== First drug (drug_name rows only) ===
Rows: 625
Resolved: 617/625 (98.7%)
Missing:  8/625 (1.3%)

=== Second drug (combos only) ===
Combo treatments: 34
Resolved: 29/34 (85.3%)
Missing:  5/34 (14.7%)

Unresolved drug names (8 unique):


['GNE-069',
 'GNE-104',
 'AZD5591',
 'BRD3379',
 'Anacardic',
 'GSK',
 'Tie2',
 'Tubastatin']

## 2. Embed a single molecule

In [ ]:
MODEL = "chemberta2MTR"

aspirin_smi = drugs["aspirin"]
assert aspirin_smi is not None, "Could not resolve aspirin to SMILES"
print(f"Aspirin SMILES: {aspirin_smi}")

emb = embedder.embed_molecule(
    identifier=aspirin_smi,
    model=MODEL,
    pooling_strategy="mean",
)

print(f"Embedding shape: {emb.shape}")
print(f"Embedding dtype: {emb.dtype}")

Aspirin SMILES: CC(=O)Oc1ccccc1C(=O)O


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Embedding shape: (384,)
Embedding dtype: float32


## 3. Batch molecule embedding

In [ ]:
smiles_list = [smi for smi in drugs.values() if smi is not None]
names_list = [n for n, s in drugs.items() if s is not None]

embeddings = embedder.embed_molecules_batch(
    identifiers=smiles_list,
    model=MODEL,
    pooling_strategy="mean",
)

for name, emb in zip(names_list, embeddings):
    if emb is not None:
        print(f"  {name:12s} → dim {emb.shape[0]}")
    else:
        print(f"  {name:12s} → FAILED")

  aspirin      → dim 384
  ibuprofen    → dim 384
  caffeine     → dim 384
  paclitaxel   → dim 384


## 4. Comparing transformer-based molecule models

Different models produce embeddings in different spaces and dimensions.

In [ ]:
import numpy as np

test_smi = drugs["caffeine"]

for m in ["chemberta2MTR", "chemberta2MLM", "molformer_base"]:
    try:
        e = embedder.embed_molecule(test_smi, model=m, pooling_strategy="mean")
        print(f"  {m:20s} → dim {e.shape[0]:,}, L2 = {np.linalg.norm(e):.2f}")
    except Exception as exc:
        print(f"  {m:20s} → {exc}")

  chemberta2MTR        → dim 384, L2 = 7.21


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-100M-MLM and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  chemberta2MLM        → dim 768, L2 = 17.30


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  molformer_base       → dim 768, L2 = 17.21


## 5. Similarity between molecules

Embedding-based cosine similarity can capture structural and functional
relatedness between drugs.

In [ ]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Embed two structurally related NSAIDs (aspirin & ibuprofen)
# and one very different molecule (caffeine)
embs = {}
for name in ["aspirin", "ibuprofen", "caffeine"]:
    smi = drugs[name]
    if smi:
        embs[name] = embedder.embed_molecule(smi, model=MODEL, pooling_strategy="mean")

if len(embs) == 3:
    print(f"aspirin  vs ibuprofen: {cosine_sim(embs['aspirin'], embs['ibuprofen']):.4f}")
    print(f"aspirin  vs caffeine:  {cosine_sim(embs['aspirin'], embs['caffeine']):.4f}")
    print(f"ibuprofen vs caffeine: {cosine_sim(embs['ibuprofen'], embs['caffeine']):.4f}")

## 6. Reverse lookup: SMILES → drug name

In [ ]:
# What is this molecule?
unknown_smi = "CC(=O)Oc1ccccc1C(O)=O"

primary = drug_resolver.smiles_to_primary_name(unknown_smi)
all_names = drug_resolver.smiles_to_names(unknown_smi, top_k=5)

print(f"Primary name: {primary}")
print(f"All names:    {all_names}")

## 8. GNN-based molecule models

`embpy` also supports **graph neural network** models that operate on the
molecular graph structure (atoms as nodes, bonds as edges), rather than
treating SMILES as a text sequence.

> **Note**: Each GNN model requires its own optional dependency. See the
> model table at the top for installation instructions.

### 8.1 MiniMol — Message-Passing GNN (GINE)

MiniMol is a 10M-parameter pre-trained **GINE** (GIN with edge features)
model trained on 6M molecules across 3,300+ bio/quantum tasks. It produces
fixed **512-dimensional** fingerprints.

```bash
pip install minimol
```

```python
# Using MiniMol through the BioEmbedder
emb_minimol = embedder.embed_molecule(
    identifier=drugs["aspirin"],
    model="minimol",
)
print(f"MiniMol embedding shape: {emb_minimol.shape}")
```

Or use the wrapper directly:

```python
from embpy.models.molecule_models import MiniMolWrapper

wrapper = MiniMolWrapper(batch_size=64)
wrapper.load(embedder.device)

emb = wrapper.embed("CCO")
print(f"Ethanol embedding: shape={emb.shape}, dtype={emb.dtype}")
# → shape=(512,), dtype=float32

# Batch embedding
batch_embs = wrapper.embed_batch(["CCO", "c1ccccc1", "CC(=O)O"])
print(f"Batch: {len(batch_embs)} embeddings of dim {batch_embs[0].shape[0]}")
```

### 8.2 MHG-GNN — Molecular Hypergraph Grammar + GNN

MHG-GNN is a **GIN-based autoencoder** pre-trained on ~1.34M PubChem molecules.
Its unique feature is that the decoder can reconstruct **structurally valid SMILES**
from latent vectors.

```bash
pip install torch-geometric huggingface-hub
pip install git+https://github.com/GT4SD/mhg-gnn.git
```

```python
from embpy.models.molecule_models import MHGGNNWrapper

wrapper = MHGGNNWrapper()
wrapper.load(embedder.device)

# Encode
emb = wrapper.embed("CCO")
print(f"Latent embedding: shape={emb.shape}")

# Batch encode
embs = wrapper.embed_batch(["CCO", "c1ccccc1"])
print(f"Batch: {len(embs)} embeddings")

# Decode — reconstruct SMILES from latent vectors
decoded = wrapper.decode(embs)
print(f"Decoded SMILES: {decoded}")
```

> **Unique capability**: MHG-GNN can **decode** latent vectors back to valid
> SMILES strings, enabling molecular generation and interpolation in
> the latent space.

### 8.3 MolE — Graph Transformer (DeBERTa-based)

MolE is a **DeBERTa-based graph transformer** pre-trained on 842M molecules from
PubChem and ZINC. It uses a CLS-token embedding for molecular representation.

> **Note**: As of writing, pretrained checkpoints for MolE have not yet been
> publicly released by the authors. You will need to obtain a checkpoint file
> before using this wrapper.

```python
from embpy.models.molecule_models import MolEWrapper

wrapper = MolEWrapper(checkpoint_path="/path/to/mole_checkpoint.pt")
wrapper.load(embedder.device)

emb = wrapper.embed("CCO")
print(f"MolE embedding: shape={emb.shape}")
```

## 9. RDKit fingerprints (non-learned baselines)

`embpy` wraps several popular **RDKit fingerprint** types through the
`RDKitWrapper`. These are deterministic (no model loading needed),
fast, and useful as baselines or for traditional ML pipelines.

```python
from embpy.models.molecule_models import RDKitWrapper

smi = "CC(=O)Oc1ccccc1C(O)=O"  # aspirin

# Binary Morgan fingerprint (ECFP4-like, 2048-dim)
rw = RDKitWrapper(fp_type="morgan_fp")
rw.load(embedder.device)
fp_binary = rw.embed(smi)
print(f"Morgan (binary): shape={fp_binary.shape}, unique values={set(fp_binary.flatten())}")

# Count-based Morgan fingerprint
rw_count = RDKitWrapper(fp_type="morgan_count_fp")
rw_count.load(embedder.device)
fp_count = rw_count.embed(smi)
print(f"Morgan (count): shape={fp_count.shape}, max count={fp_count.max():.0f}")

# MACCS keys (167-dim)
rw_maccs = RDKitWrapper(fp_type="maccs_fp")
rw_maccs.load(embedder.device)
fp_maccs = rw_maccs.embed(smi)
print(f"MACCS: shape={fp_maccs.shape}")
```

Available fingerprint types:
- `rdkit_fp` — RDKit topological (binary, 2048-dim)
- `morgan_fp` / `morgan_count_fp` — Morgan/ECFP (binary or count, 2048-dim)
- `maccs_fp` — MACCS keys (binary, 167-dim)
- `atom_pair_fp` / `atom_pair_count_fp` — Atom pair (binary or count, 2048-dim)
- `torsion_fp` / `torsion_count_fp` — Topological torsion (binary or count, 2048-dim)

## 10. Building an embedding matrix from an AnnData

If you have a perturbation-screen `AnnData` whose `.obs` contains a
column with drug names or SMILES, use
`PerturbationProcessor.build_molecule_embedding_matrix` to embed them
all at once. It resolves names → SMILES automatically, embeds, and
stores the result in `.obsm`.

This works with **any** of the models above — transformer, GNN, or RDKit.

In [ ]:
import anndata as ad
import pandas as pd
from embpy.pp.basic import PerturbationProcessor

pp = PerturbationProcessor(embedder=embedder)

# Simulate a screen AnnData with drug metadata
screen_obs = pd.DataFrame({
    "treatment":  ["aspirin", "ibuprofen", "caffeine", "paclitaxel"],
    "dose_uM":    [10.0, 5.0, 1.0, 0.1],
    "cell_line":  ["A549", "A549", "HeLa", "HeLa"],
})
adata_screen = ad.AnnData(obs=screen_obs)

# Embed – just point at the .obs column with drug names
adata_emb = pp.build_molecule_embedding_matrix(
    adata=adata_screen,
    column="treatment",        # which .obs column has drug identifiers
    id_type="drug_name",       # they are common names, not SMILES
    model="chemberta2MTR",
    obsm_key="X_chemberta",
)

print("Result AnnData (original metadata preserved):")
print(adata_emb.obs)
print(f"\nEmbedding matrix: {adata_emb.obsm['X_chemberta'].shape}")
print(f"Successful: {adata_emb.obs['embedded'].sum()} / {len(adata_emb)}")

## Summary

### Resolve & embed

| What | How |
|---|---|
| Drug name → SMILES | `DrugResolver().name_to_smiles("aspirin")` |
| SMILES → drug name | `DrugResolver().smiles_to_names(smiles)` |
| Single molecule embedding | `embed_molecule(smiles, model)` |
| Batch molecule embedding | `embed_molecules_batch(smiles_list, model)` |
| Embed from AnnData column | `pp.build_molecule_embedding_matrix(adata=..., column="treatment")` |
| Embed from a list | `pp.build_molecule_embedding_matrix(identifiers=[...])` |

### Available model categories

| Category | Models | Key strengths |
|---|---|---|
| **Transformer** | ChemBERTa-2 (MTR/MLM), MolFormer | Large-scale pretraining on SMILES strings |
| **Message-Passing GNN** | MiniMol (GINE) | Lightweight, 512-dim graph-level fingerprints |
| **Graph Autoencoder** | MHG-GNN | Encode **and decode** — reconstruct valid SMILES from latent vectors |
| **Graph Transformer** | MolE | DeBERTa-based, pre-trained on 842M molecules |
| **RDKit fingerprints** | Morgan, MACCS, atom-pair, torsion, RDKit-FP | Deterministic, fast, binary or count-based |

Next: [Tutorial 5 – Text Embeddings](05_text_embeddings.ipynb)